## 프롬프트 테스트

<div style="text-align: right"> <b>KMYU</b></div>
<div style="text-align: right"> Initial issue : 2026.03.20 </div>
<div style="text-align: right"> last update : 2026.03.20 </div>

개정 이력  
- `2026.03.20` : 노트북 최초 작성

In [ ]:
import sys

sys.path.insert(0, "../../")

from dotenv import load_dotenv

load_dotenv()

### 1. 프롬프트 로더 확인

In [ ]:
from backend.app.agent.prompt_loader import load_prompt, load_system_prompt

# 전체 프롬프트 YAML 목록 로드 확인
prompt_names = ["system", "deal_structuring", "scoring", "resource_estimation", "risk_analysis", "similar_project", "final_verdict"]

for name in prompt_names:
    tpl = load_prompt(name)
    print(f"[{name}] version={tpl.version}, has_output_schema={tpl.output_schema is not None}")

In [ ]:
# 프롬프트 구조 상세 확인 (deal_structuring 예시)
tpl = load_prompt("deal_structuring")
print("=== System Prompt (raw template) ===")
print(tpl._system_prompt[:200], "...")
print("\n=== User Prompt (raw template) ===")
print(tpl._user_prompt[:300], "...")
print("\n=== Output Schema ===")
print(tpl.output_schema)

### 2. system.yaml 렌더링

In [ ]:
# system.yaml에 주입할 변수 준비
sample_company_context = """[사업 방향] AI 솔루션 개발 전문기업, 자연어처리 및 컴퓨터비전 분야 핵심 역량 보유
[단기 전략] 2026년 상반기 B2B SaaS 전환 가속화"""

sample_deal_criteria = "계약 금액 5천만원 이상, 수행 기간 6개월 이내, 자사 기술 스택 활용 가능한 프로젝트 우선"

sample_scoring_criteria = [
    {"name": "기술 적합성", "weight": 20.0, "description": "자사 기술 스택과의 적합도"},
    {"name": "수익성", "weight": 20.0, "description": "예상 수익 대비 비용 효율"},
    {"name": "리소스 가용성", "weight": 15.0, "description": "현재 투입 가능 인력 수준"},
    {"name": "납기 리스크", "weight": 15.0, "description": "일정 준수 가능성"},
    {"name": "고객 리스크", "weight": 10.0, "description": "고객사 안정성 및 협업 리스크"},
    {"name": "요구사항 명확성", "weight": 10.0, "description": "요구사항의 명확도"},
    {"name": "전략적 가치", "weight": 10.0, "description": "중장기 전략과의 정합성"},
]

In [ ]:
system_tpl = load_system_prompt()
system_base = system_tpl.render_system(
    company_context=sample_company_context,
    deal_criteria=sample_deal_criteria,
    scoring_criteria=sample_scoring_criteria,
)
print(system_base)

### 3. deal_structuring 프롬프트 → LLM 호출

In [ ]:
from backend.app.agent.base import call_llm, parse_json_response

# 샘플 딜 입력 텍스트
sample_deal_input = """고객사: 메가테크 (대기업, 제조업)
프로젝트: 공장 설비 예지보전 AI 시스템 개발
배경: 기존 정기 점검 체계에서 AI 기반 예측 유지보수로 전환하여 설비 가동률 향상 및 유지보수 비용 절감
기술 요구사항: Python, TensorFlow, 시계열 분석, IoT 센서 데이터 처리, 대시보드 (React)
예상 규모: 2억원
기간: 5개월
결제 조건: 착수금 30%, 중간 40%, 완료 30%
특이사항: 공장 내 보안 네트워크 환경에서만 운영, 온프레미스 배포 필수"""

In [ ]:
# deal_structuring 프롬프트 렌더링
tpl = load_prompt("deal_structuring")
system_prompt, user_prompt = tpl.render(
    system_base=system_base,
    deal_input=sample_deal_input,
)

print("=== System Prompt ===")
print(system_prompt[:500], "...")
print("\n=== User Prompt ===")
print(user_prompt)

In [ ]:
# LLM 호출 및 파싱
raw_output = await call_llm(system_prompt, user_prompt)
print("=== LLM Raw Output ===")
print(raw_output)

In [ ]:
structured_deal = parse_json_response(raw_output)

import json

print("=== Parsed Structured Deal ===")
print(json.dumps(structured_deal, ensure_ascii=False, indent=2))

### 4. scoring 프롬프트 → LLM 호출

In [ ]:
tpl = load_prompt("scoring")
system_prompt, user_prompt = tpl.render(
    system_base=system_base,
    structured_deal=structured_deal,
    scoring_criteria=sample_scoring_criteria,
    company_context=sample_company_context,
)

print("=== User Prompt (scoring) ===")
print(user_prompt[:800], "...")

In [ ]:
raw_output = await call_llm(system_prompt, user_prompt)
scoring_result = parse_json_response(raw_output)

print("=== Scoring Result ===")
print(json.dumps(scoring_result, ensure_ascii=False, indent=2))

### 5. resource_estimation 프롬프트 → 렌더링 확인

In [ ]:
sample_team_members = [
    {"name": "김개발", "role": "MLE", "monthly_rate": 800, "is_available": True, "current_project": None, "available_from": None},
    {"name": "이기획", "role": "PM", "monthly_rate": 900, "is_available": True, "current_project": None, "available_from": None},
    {"name": "박엔지", "role": "Backend", "monthly_rate": 750, "is_available": False, "current_project": "프로젝트A", "available_from": "2026-05-01"},
]

sample_company_rates = "MLE: 월 800만원, PM: 월 900만원, Backend: 월 750만원, Frontend: 월 700만원"
sample_past_projects = []  # 벡터 스토어 없이 빈 리스트로 테스트

In [ ]:
tpl = load_prompt("resource_estimation")
system_prompt, user_prompt = tpl.render(
    system_base=system_base,
    structured_deal=structured_deal,
    team_members=sample_team_members,
    company_rates=sample_company_rates,
    past_projects=sample_past_projects,
)

print("=== User Prompt (resource_estimation) ===")
print(user_prompt)

In [ ]:
raw_output = await call_llm(system_prompt, user_prompt)
resource_result = parse_json_response(raw_output)

print("=== Resource Estimation Result ===")
print(json.dumps(resource_result, ensure_ascii=False, indent=2))

### 6. risk_analysis 프롬프트 → LLM 호출

In [ ]:
tpl = load_prompt("risk_analysis")
system_prompt, user_prompt = tpl.render(
    system_base=system_base,
    structured_deal=structured_deal,
    company_context=sample_company_context,
)

raw_output = await call_llm(system_prompt, user_prompt)
risk_result = parse_json_response(raw_output)

print("=== Risk Analysis Result ===")
print(json.dumps(risk_result, ensure_ascii=False, indent=2))

### 7. final_verdict 프롬프트 → 마크다운 리포트

In [ ]:
tpl = load_prompt("final_verdict")
system_prompt, user_prompt = tpl.render(
    system_base=system_base,
    structured_deal=structured_deal,
    scores=scoring_result.get("scores", []),
    total_score=scoring_result.get("total_score", 0),
    verdict=scoring_result.get("verdict", "pending"),
    resource_estimate=resource_result,
    risks=risk_result.get("risks", []),
    similar_projects=[],
)

print("=== User Prompt (final_verdict) ===")
print(user_prompt[:1000], "...")

In [ ]:
final_report = await call_llm(system_prompt, user_prompt)

print("=== Final Verdict Report (Markdown) ===")
print(final_report)

In [ ]:
# 마크다운 렌더링 확인
from IPython.display import Markdown, display

display(Markdown(final_report))